## Imports

In [ ]:
import cv2
import numpy as np
from patchify import patchify
from keras.utils import normalize
import tensorflow as tf
from tensorflow.keras import backend as K
import matplotlib.pyplot as plt

## Image path
Set `IMG_PATH` and `MAGNIFICATION` before running.

In [ ]:
IMG_PATH     = r"C:\Users\mazurm\deepaxon\1502_RUNDist_100X_001.tif"
MAGNIFICATION = '100x'   # '40x' or '100x'

## Core utilities

In [ ]:
TARGET_SIZE = (1440, 1024)  # (width, height) for cv2

def resize_img(img_path: str, is_mask: bool = False) -> np.ndarray:
    """Load and resize an image to TARGET_SIZE (1440x1024)."""
    img = cv2.imread(img_path, 0)
    if img is None:
        raise ValueError(f"Could not read image: {img_path}")
    h, w = img.shape[:2]
    if (w, h) == TARGET_SIZE:
        return img
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_AREA
    return cv2.resize(img, TARGET_SIZE, interpolation=interp)


def center_crop(img, patch_size):
    h, w = img.shape[:2]
    crop_h = (h // patch_size) * patch_size
    crop_w = (w // patch_size) * patch_size
    start_h = (h - crop_h) // 2
    start_w = (w - crop_w) // 2
    return img[start_h:start_h + crop_h, start_w:start_w + crop_w]

In [ ]:
def _hann_fn(x, patch_size=256):
    return (1 - np.cos(2 * np.pi * x / (patch_size - 1))) / 2


def get_pos(shape, i, j):
    i_max = shape[0] - 1
    j_max = shape[1] - 1
    if   i == 0     and j == 0:     return 0
    elif i == 0     and j == j_max: return 2
    elif i == i_max and j == 0:     return 6
    elif i == i_max and j == j_max: return 8
    elif i == 0:                    return 1
    elif i == i_max:                return 7
    elif j == 0:                    return 3
    elif j == j_max:                return 5
    else:                           return 4


def hann_window(pos, patch_size=256):
    half = patch_size // 2
    i, j = np.meshgrid(np.arange(patch_size), np.arange(patch_size), indexing='ij')
    c1 = (i <= half) & (j <= half)
    c2 = (i > half)  & (j <  half)
    c3 = (i <  half) & (j >  half)
    c4 = ~c1 & ~c2 & ~c3
    s  = np.zeros((patch_size, patch_size), dtype=float)
    hi = _hann_fn(i.astype(float), patch_size)
    hj = _hann_fn(j.astype(float), patch_size)
    if pos == 0:
        s[c1]=1;             s[c2]=hi[c2];          s[c3]=hj[c3];          s[c4]=hi[c4]*hj[c4]
    elif pos == 1:
        s[c1]=hj[c1];        s[c2]=hi[c2]*hj[c2];   s[c3]=hj[c3];          s[c4]=hi[c4]*hj[c4]
    elif pos == 2:
        s[c1]=hj[c1];        s[c2]=hi[c2]*hj[c2];   s[c3]=1;               s[c4]=hi[c4]
    elif pos == 3:
        s[c1]=hi[c1];        s[c2]=hi[c2];           s[c3]=hi[c3]*hj[c3];   s[c4]=hi[c4]*hj[c4]
    elif pos == 4:
        s[c1]=hi[c1]*hj[c1]; s[c2]=hi[c2]*hj[c2];   s[c3]=hi[c3]*hj[c3];   s[c4]=hi[c4]*hj[c4]
    elif pos == 5:
        s[c1]=hi[c1]*hj[c1]; s[c2]=hi[c2]*hj[c2];   s[c3]=hi[c3];          s[c4]=hi[c4]
    elif pos == 6:
        s[c1]=hi[c1];        s[c2]=1;                s[c3]=hi[c3]*hj[c3];   s[c4]=hj[c4]
    elif pos == 7:
        s[c1]=hi[c1]*hj[c1]; s[c2]=hj[c2];          s[c3]=hi[c3]*hj[c3];   s[c4]=hj[c4]
    elif pos == 8:
        s[c1]=hi[c1]*hj[c1]; s[c2]=hj[c2];          s[c3]=hi[c3];          s[c4]=1
    return s


def recolor(pred):
    """BGW: 0=black, 1=grey(128), 2=white(255)"""
    out = np.zeros(pred.shape, dtype=np.uint8)
    out[pred == 1] = 128
    out[pred == 2] = 255
    return np.stack([out, out, out], axis=-1)

## Enhancement pipeline
`enhance_patch` replaces the old `clahe` and `patch_clahe` functions.

**Workflow per 256px patch:**
1. OpenCV CLAHE — zone-driven `clip_limit` (redistributes histogram)
2. `patch_clahe_fast` — vectorized pixel-level brightness correction within the patch
3. Unsharp mask fine (σ scales with small axon rings)
4. Unsharp mask coarse (σ scales with large axon rings)
5. Zone gate — blends enhanced vs original by patch brightness

**Magnification-aware sigmas (after resize to 1280×1024):**
- 40×: `sigma_fine=1px`, `sigma_coarse=6px`
- 100×: `sigma_fine=2px`, `sigma_coarse=15px`

In [ ]:
def patch_clahe_fast(img, lower, upper, img_mean, img_std):
    """
    Vectorized pixel-level brightness correction.
    Replaces the old slow patch_clahe loop.
    Same zone logic as before but computed as numpy arrays — no Python loop.
    """
    img_f = img.astype(np.float32)

    # Local brightness estimate per pixel via 3x3 mean filter
    kernel = np.ones((3, 3), dtype=np.float32) / 9.0
    brightness_map = cv2.filter2D(img_f, -1, kernel)

    alpha_map = np.ones_like(brightness_map)

    below_mask = (brightness_map < img_mean) & (brightness_map >= lower)
    above_mask = (brightness_map > img_mean) & (brightness_map <= upper)

    # Safe log: only compute on masked pixels — avoids log(0)
    z_below = np.ones_like(brightness_map)
    z_above = np.ones_like(brightness_map)

    z_below[below_mask] = np.clip(
        (img_mean - brightness_map[below_mask]) / img_std, 1e-6, 0.499
    )
    z_above[above_mask] = np.clip(
        (brightness_map[above_mask] - img_mean) / img_std, 1e-6, 0.499
    )

    alpha_map[below_mask] = 1 / (2 * (1 - np.log(z_below[below_mask])))
    alpha_map[above_mask] = (1 - np.log(z_above[above_mask])) / 2

    # Hard clip alpha to prevent blow-out
    alpha_map = np.clip(alpha_map, 0.5, 2.0)

    return np.clip(img_f * alpha_map, 0, 255).astype(np.uint8)


def enhance_patch(patch, lower, upper, img_mean, img_std, magnification='40x',
                  clip_min=1.0, clip_max=2.0):
    """
    Full enhancement pipeline for a single 256px patch.

    Steps:
      1. Zone-driven CLAHE   — clip_limit scales with patch brightness distance from mean
      2. patch_clahe_fast    — vectorized pixel-level correction
      3. Fine unsharp mask   — targets small axon rings
      4. Coarse unsharp mask — targets large axon rings
      5. Zone blend          — controls how much enhancement reaches the model
    """
    # Magnification-aware sigmas
    if magnification == '100x':
        sigma_fine, sigma_coarse = 2.0, 15.0
    else:  # 40x default
        sigma_fine, sigma_coarse = 1.0, 6.0

    brightness = np.median(patch)

    # ── Zone position: t=0 at mean (no correction), t=1 at boundary (full correction) ──
    if brightness < lower or brightness > upper:
        t = 0.0          # outside zone — leave untouched
        clip_limit = 0.0
    elif brightness < img_mean:
        t = (img_mean - brightness) / (img_mean - lower)
        clip_limit = clip_min + (clip_max - clip_min) * t
    else:
        t = (brightness - img_mean) / (upper - img_mean)
        clip_limit = clip_min + (clip_max - clip_min) * t

    # ── Step 1: CLAHE (histogram redistribution) ──
    if clip_limit > 0:
        clahe_op = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
        enhanced = clahe_op.apply(patch)
    else:
        enhanced = patch.copy()

    # ── Step 2: patch_clahe_fast (pixel-level correction) ──
    if clip_limit > 0:
        enhanced = patch_clahe_fast(enhanced, lower, upper, img_mean, img_std)

    # ── Step 3: Fine unsharp mask (small rings) ──
    amount_fine = 0.3 + 0.5 * t   # 0.3 at mean → 0.8 at boundary
    blur_fine   = cv2.GaussianBlur(enhanced.astype(np.float32), (0, 0), sigma_fine)
    enhanced    = np.clip(
        enhanced.astype(np.float32) * (1 + amount_fine) - blur_fine * amount_fine,
        0, 255
    ).astype(np.uint8)

    # ── Step 4: Coarse unsharp mask (large rings) ──
    amount_coarse = 0.2 + 0.3 * t  # 0.2 at mean → 0.5 at boundary
    blur_coarse   = cv2.GaussianBlur(enhanced.astype(np.float32), (0, 0), sigma_coarse)
    enhanced      = np.clip(
        enhanced.astype(np.float32) * (1 + amount_coarse) - blur_coarse * amount_coarse,
        0, 255
    ).astype(np.uint8)

    # ── Step 5: Zone blend — t=0 (at mean) → mostly original, t=1 (boundary) → mostly enhanced ──
    blend_enhanced = 0.3 + 0.7 * t   # 0.3 at mean → 1.0 at boundary
    blend_original = 1.0 - blend_enhanced
    result = cv2.addWeighted(enhanced, blend_enhanced, patch, blend_original, 0)

    return result

## Loss functions and model

In [ ]:
def dice_coef(y_true, y_pred, smooth: float = 1.0):
    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(tf.cast(y_pred, tf.float32))
    intersection = K.sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

def iou_coef(y_true, y_pred, smooth: float = 1.0):
    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(tf.cast(y_pred, tf.float32))
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def combined_loss(y_true, y_pred):
    """Weighted combination of dice loss and binary cross-entropy."""
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return 0.5 * dice_loss(y_true, y_pred) + 0.5 * K.mean(bce)

custom_objects = {
    'dice_coef':     dice_coef,
    'dice_loss':     dice_loss,
    'iou_coef':      iou_coef,
    'combined_loss': combined_loss,
}

model = tf.keras.models.load_model(
    r"C:\Users\mazurm\deepaxon\models\rb_v1.keras",
    custom_objects=custom_objects,
    compile=False
)

## Segmentation functions

In [ ]:
def segment_image(img_path, model, patch_size=256):
    """Baseline segmentation — no preprocessing."""
    step = patch_size // 2

    img      = resize_img(img_path, is_mask=False)
    img_crop = center_crop(img, patch_size)
    crop_h, crop_w = img_crop.shape[:2]

    patches    = patchify(img_crop, (patch_size, patch_size), step=step)
    n_rows, n_cols = patches.shape[:2]
    pred_img   = np.zeros((crop_h, crop_w), dtype=float)

    for i in range(n_rows):
        for j in range(n_cols):
            patch = patches[i, j, :, :]
            patch = normalize(patch)
            patch = np.expand_dims(patch, axis=(0, 3))
            pred  = model.predict(patch, verbose=0)
            pred  = np.argmax(pred, axis=3)[0]

            pos  = get_pos((n_rows, n_cols), i, j)
            hann = hann_window(pos, patch_size)
            i_start = i * step
            j_start = j * step
            pred_img[i_start:i_start + patch_size,
                     j_start:j_start + patch_size] += pred * hann

    pred_img = np.round(pred_img).astype(int)
    return recolor(pred_img), img_crop

In [ ]:
def enhanced_segment_image(img_path, model, magnification=None, patch_size=256,
                            clip_min=1.0, clip_max=2.0):
    """
    Segmentation with enhance_patch preprocessing.
    Returns: (pred_raw, pred_enhanced, enhanced_img, img_crop)
    """
    # Use global MAGNIFICATION if not passed explicitly
    mag = magnification or MAGNIFICATION

    step = patch_size // 2

    img      = resize_img(img_path, is_mask=False)
    img_crop = center_crop(img, patch_size)
    crop_h, crop_w = img_crop.shape[:2]

    # Image-level stats for zone gate
    lower    = np.mean(img_crop) - (np.std(img_crop) / 2)
    upper    = np.mean(img_crop) + (np.std(img_crop) / 2)
    img_mean = np.mean(img_crop)
    img_std  = np.std(img_crop)

    patches    = patchify(img_crop, (patch_size, patch_size), step=step)
    n_rows, n_cols = patches.shape[:2]

    pred_raw_img  = np.zeros((crop_h, crop_w), dtype=float)
    pred_enh_img  = np.zeros((crop_h, crop_w), dtype=float)
    enhanced_full = np.zeros((crop_h, crop_w), dtype=float)

    for i in range(n_rows):
        for j in range(n_cols):
            patch = patches[i, j, :, :]

            # ── Raw prediction ──
            p_raw  = normalize(patch)
            p_raw  = np.expand_dims(p_raw, axis=(0, 3))
            pr     = model.predict(p_raw, verbose=0)
            pr     = np.argmax(pr, axis=3)[0]

            # ── Enhanced prediction ──
            enh    = enhance_patch(patch, lower, upper, img_mean, img_std,
                                   magnification=mag,
                                   clip_min=clip_min, clip_max=clip_max)
            p_enh  = normalize(enh)
            p_enh  = np.expand_dims(p_enh, axis=(0, 3))
            pe     = model.predict(p_enh, verbose=0)
            pe     = np.argmax(pe, axis=3)[0]

            pos  = get_pos((n_rows, n_cols), i, j)
            hann = hann_window(pos, patch_size)
            i_start = i * step
            j_start = j * step

            pred_raw_img [i_start:i_start + patch_size,
                          j_start:j_start + patch_size] += pr  * hann
            pred_enh_img [i_start:i_start + patch_size,
                          j_start:j_start + patch_size] += pe  * hann
            enhanced_full[i_start:i_start + patch_size,
                          j_start:j_start + patch_size] += enh * hann

    pred_raw = recolor(np.round(pred_raw_img).astype(int))
    pred_enh = recolor(np.round(pred_enh_img).astype(int))

    return pred_raw, pred_enh, enhanced_full, img_crop

## Design cells
Run these independently to tune parameters — **no model calls**.
Run `Kernel → Restart & Run All` first to make sure everything is in memory.

In [ ]:
# ── Patch stats and grid info ──────────────────────────────────────────────
img      = resize_img(IMG_PATH, is_mask=False)
img_crop = center_crop(img, patch_size=256)

lower    = np.mean(img_crop) - (np.std(img_crop) / 2)
upper    = np.mean(img_crop) + (np.std(img_crop) / 2)
img_mean = np.mean(img_crop)
img_std  = np.std(img_crop)

patches  = patchify(img_crop, (256, 256), step=128)

print(f"Patch grid : {patches.shape[0]} rows x {patches.shape[1]} cols")
print(f"img_mean   : {img_mean:.1f}")
print(f"img_std    : {img_std:.1f}")
print(f"lower      : {lower:.1f}")
print(f"upper      : {upper:.1f}")
print()

# Zone coverage diagnostic
in_zone, out_zone = 0, 0
for i in range(patches.shape[0]):
    for j in range(patches.shape[1]):
        b = np.median(patches[i, j])
        if lower <= b <= upper:
            in_zone += 1
        else:
            out_zone += 1
total = in_zone + out_zone
print(f"In zone    : {in_zone}/{total}  ({100*in_zone/total:.0f}%)")
print(f"Out of zone: {out_zone}/{total}  ({100*out_zone/total:.0f}%)")


In [ ]:
# ── Single patch enhance_patch sweep ───────────────────────────────────────
# Adjust i, j to a patch of interest from the grid printed above
i, j = 2, 3
test_patch = patches[i, j, :, :]

clip_maxes = [1.5, 2.0, 2.5, 3.0]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for col, cmax in enumerate(clip_maxes):
    enhanced = enhance_patch(test_patch, lower, upper, img_mean, img_std,
                             magnification=MAGNIFICATION,
                             clip_min=1.0, clip_max=cmax)
    axes[0, col].imshow(test_patch, 'gray', vmin=0, vmax=255)
    axes[0, col].set_title('Original')
    axes[0, col].axis('off')
    axes[1, col].imshow(enhanced, 'gray', vmin=0, vmax=255)
    axes[1, col].set_title(f'clip_max={cmax}')
    axes[1, col].axis('off')

brightness = np.median(test_patch)
t = (img_mean - brightness) / (img_mean - lower) if brightness < img_mean     else (brightness - img_mean) / (upper - img_mean)
t = max(0.0, min(1.0, t))

plt.suptitle(f'Patch [{i},{j}]  median={brightness:.1f}  t={t:.2f}  mag={MAGNIFICATION}',
             y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── patch_clahe_fast standalone check ──────────────────────────────────────
# Verifies the vectorized pixel correction alone on a single patch
i, j = 2, 3
test_patch = patches[i, j, :, :]

fast_result = patch_clahe_fast(test_patch, lower, upper, img_mean, img_std)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(test_patch, 'gray', vmin=0, vmax=255)
axes[0].set_title(f'Original  median={np.median(test_patch):.1f}')
axes[0].axis('off')
axes[1].imshow(fast_result, 'gray', vmin=0, vmax=255)
axes[1].set_title(f'patch_clahe_fast  median={np.median(fast_result):.1f}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Full segmentation comparison
Runs both raw and enhanced segmentation and compares side by side.
**This cell calls the model — takes ~15–20s.**

In [ ]:
pred_raw, pred_enh, enh_img, img_crop = enhanced_segment_image(
    IMG_PATH, model,
    magnification=MAGNIFICATION,
    clip_min=1.0,
    clip_max=2.0       # adjust based on sweep results above
)

fig, axes = plt.subplots(1, 4, figsize=(24, 6))
axes[0].imshow(img_crop, 'gray')
axes[0].set_title('Original image')
axes[0].axis('off')
axes[1].imshow(enh_img, 'gray')
axes[1].set_title('Enhanced image')
axes[1].axis('off')
axes[2].imshow(pred_raw)
axes[2].set_title('Segmentation — raw')
axes[2].axis('off')
axes[3].imshow(pred_enh)
axes[3].set_title('Segmentation — enhanced')
axes[3].axis('off')
plt.tight_layout()
plt.show()

## Segmentation difference overlay
Blue = predicted by raw only. Green = predicted by enhanced only.

In [ ]:
gray_raw = cv2.cvtColor(pred_raw, cv2.COLOR_BGR2GRAY)
gray_enh = cv2.cvtColor(pred_enh, cv2.COLOR_BGR2GRAY)

diff = gray_raw.astype(np.int16) - gray_enh.astype(np.int16)
thresh = 30
kernel = np.ones((3, 3), np.uint8)

mask_raw_only = cv2.morphologyEx((diff >  thresh).astype(np.uint8), cv2.MORPH_OPEN, kernel).astype(bool)
mask_enh_only = cv2.morphologyEx((diff < -thresh).astype(np.uint8), cv2.MORPH_OPEN, kernel).astype(bool)

overlay = pred_raw.copy()
overlay[mask_raw_only] = [0,   0, 255]   # blue  = raw only
overlay[mask_enh_only] = [0, 255,   0]   # green = enhanced only
result = cv2.addWeighted(pred_raw, 0.7, overlay, 0.3, 0)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
axes[0].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
axes[0].set_title('Diff overlay  (blue=raw only  green=enhanced only)')
axes[0].axis('off')
axes[1].imshow(img_crop, 'gray')
axes[1].set_title('Original image')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f"Pixels raw only    : {mask_raw_only.sum()}")
print(f"Pixels enhanced only: {mask_enh_only.sum()}")